# V1-21: LangChain Tool Calling Prompt 비교

## 목적

같은 OpenAI 모델·동일 Tool 스키마에서, **간단한 Router Prompt(A)** 와 **V1 데이터·기간·Tool 경계를 명시한 Prompt(B)** 가 사용자 질문에 필요한 Tool 호출 요청과 인자를 얼마나 정확히 만드는지 비교한다.

이번 notebook은 실제 OpenDART를 조회하거나 Tool 함수를 실행하지 않는다. `bind_tools()`를 받은 LLM이 반환한 `tool_calls`(실행 주문서)만 측정한다. 따라서 이 결과는 **Tool 선택·인자 생성 품질**을 보는 LangChain 평가이며, 실제 Tool 실행·계산·답변 품질 평가는 다음 단계다.

- 고정 조건: `.env`의 `OPENAI_MODEL`, `temperature=0`, `reasoning_effort='none'`, 같은 Tool 정의·같은 17개 개발 질문
- 비교 대상: A. 짧은 Router 지시 / B. 상세 정책 Router 지시
- 핵심 지표: Tool Precision·Recall·F1, Tool-set Exact Match, Argument Exact Match(계약 필드), Invalid-plan Rate, API 호출 성공률, 지연시간·token usage
- 제외: 기업명 `corp_code` 확정, 실제 OpenDART 호출, 최근 연도의 실제 가용성 판정, 재무 계산, RAG 검색, 최종 답변

> 정답은 `EVAL_CASES`에만 저장하며 모델 Prompt에는 넣지 않는다. `최근 N년`은 실제 연도로 임의 변환하지 않고 `recent_years=N`으로 보존한다. 실제 가용 연도 선택은 이후 Period Resolver의 책임이다.


In [ ]:
from datetime import datetime, timezone
from importlib.metadata import version
import json
import os
from pathlib import Path
import sys
import time
from typing import Literal

import pandas as pd
from langchain.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI


In [ ]:
current_dir = Path.cwd().resolve()
if current_dir.name == 'v1':
    project_root = current_dir.parents[1]
elif current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

env_path = project_root / '.env'
if not env_path.exists():
    raise FileNotFoundError(f'{env_path} 파일을 먼저 만들어 주세요.')

for raw_line in env_path.read_text(encoding='utf-8').splitlines():
    line = raw_line.strip()
    if line and not line.startswith('#') and '=' in line:
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip(chr(39) + chr(34)))

if not os.environ.get('OPENAI_API_KEY'):
    raise ValueError('.env에 OPENAI_API_KEY를 설정해 주세요.')

OPENAI_MODEL = os.environ.get('OPENAI_MODEL', 'gpt-5.6-luna')
TEMPERATURE = 0
REQUEST_TIMEOUT_S = 120
MAX_RETRIES = 0

RUN_CONFIG = {
    'run_at_utc': datetime.now(timezone.utc).isoformat(),
    'model': OPENAI_MODEL,
    'temperature': TEMPERATURE,
    'reasoning_effort': 'none',
    'timeout_s': REQUEST_TIMEOUT_S,
    'max_retries': MAX_RETRIES,
    'langchain': version('langchain'),
    'langchain_openai': version('langchain-openai'),
}
print(f'Project root: {project_root}')
print(f'OpenAI API key loaded: {bool(os.environ.get("OPENAI_API_KEY"))} (값은 출력하지 않음)')
RUN_CONFIG


In [ ]:
# 아래 함수는 이번 평가에서 실행하지 않는다. 함수명·설명·인자 스키마를 LLM에 전달하는 Tool 계약이다.
@tool
def get_financial_data(
    companies: list[str],
    metrics: list[str],
    years: list[int] | None = None,
    recent_years: int | None = None,
    statement_basis: Literal['DEFAULT', 'CFS', 'OFS'] = 'DEFAULT',
) -> str:
    """OpenDART 연간 재무 원천값을 조회한다. 매출액(revenue), 영업이익(operating_income), 당기순이익(net_income), 자산(assets), 부채(liabilities), 자본(equity), 유동자산(current_assets), 유동부채(current_liabilities), 영업활동현금흐름(operating_cash_flow)만 요청한다. 명시 연도는 years, '최근 N년'은 recent_years에 넣는다. 기준을 사용자가 명시하지 않으면 DEFAULT를 사용하며, 실제 CFS 우선/OFS 전체 대체는 조회 코드가 결정한다."""
    return '계획 평가용 Tool입니다. 실제 OpenDART 호출은 하지 않습니다.'


@tool
def calculate_financial_metric(
    companies: list[str],
    calculations: list[str],
    years: list[int] | None = None,
    recent_years: int | None = None,
) -> str:
    """재무 원천값을 받은 뒤 파생 지표를 계산한다. operating_margin(영업이익률), revenue_growth(매출 성장률), debt_ratio(부채비율)만 계산한다. 원천 재무값 조회가 필요한 질문에서는 get_financial_data도 함께 요청해야 한다."""
    return '계획 평가용 Tool입니다. 실제 계산은 하지 않습니다.'


@tool
def search_disclosure_list(
    companies: list[str],
    years: list[int] | None = None,
    recent_years: int | None = None,
    keywords: list[str] | None = None,
) -> str:
    """OpenDART 공시 목록을 기간과 선택 키워드로 조회한다. 사용자가 '공시 목록', '최근 공시', 특정 공시의 날짜·제목을 요구할 때만 사용한다. 사업보고서 본문의 위험요인·사업 내용 근거 검색에는 사용하지 않는다."""
    return '계획 평가용 Tool입니다. 실제 공시 목록 호출은 하지 않습니다.'


@tool
def search_business_report_evidence(
    companies: list[str],
    topics: list[str],
    years: list[int] | None = None,
    recent_years: int | None = None,
) -> str:
    """사업보고서 원문에서 위험요인, 사업 내용, 투자 계획처럼 문장 근거가 필요한 주제를 검색한다. 공시 목록 자체나 재무 숫자만 요구하는 질문에는 사용하지 않는다. 이 Tool의 문서 parsing·RAG 구현은 이후 단계다."""
    return '계획 평가용 Tool입니다. 실제 사업보고서 검색은 하지 않습니다.'


TOOLS = [get_financial_data, calculate_financial_metric, search_disclosure_list, search_business_report_evidence]
TOOL_NAMES = {tool_.name for tool_ in TOOLS}
pd.DataFrame([{'Tool': tool_.name, '설명': tool_.description.split('.')[0]} for tool_ in TOOLS])


In [ ]:
# Prompt A는 최소 지시 기준선, Prompt B는 V1의 데이터·기간·안전 경계를 명시한 후보다.
PROMPT_A = '''
당신은 FinSight의 Tool Router다. 사용자 질문을 읽고 필요한 Tool만 호출 요청으로 반환한다.
Tool 설명과 인자 스키마를 따른다. 실제 데이터를 만들거나 답변하지 않는다.
지원하지 않는 요청에는 Tool을 호출하지 않는다.
'''

PROMPT_B = '''
역할: 당신은 국내 기업 리서치 시스템 FinSight의 Tool Router다. 답변을 만들거나 데이터를 추측하지 말고, 사용자 요청을 처리하는 데 필요한 Tool 호출 요청만 만든다.

공통 규칙:
1. 질문에 실제로 언급된 기업 표현만 companies에 넣는다. 계열사·경쟁사·유사 기업을 추가하지 않는다. 기업명 해석과 corp_code 확정은 별도 기업 식별 경로의 책임이다.
2. 명시 연도·범위(예: 2022~2024)는 years의 오름차순 목록으로 넣는다. '최근 N년'은 실제 연도를 추측하지 말고 years=[]와 recent_years=N으로 보존한다. 실제 가용 연도 선택은 Period Resolver가 한다.
3. 매출액·영업이익·당기순이익·자산·부채·자본·유동자산·유동부채·영업활동현금흐름은 get_financial_data의 canonical metric으로 넣는다.
4. 영업이익률은 operating_margin, 매출 성장률은 revenue_growth, 부채비율은 debt_ratio다. 파생 지표를 요구하면 get_financial_data와 calculate_financial_metric을 모두 요청한다. calculator만 단독 호출하지 않는다.
5. '공시 목록/최근 공시/공시 제목·날짜'는 search_disclosure_list다. 사업보고서 본문에서 위험요인·사업 내용·투자계획 같은 문장 근거를 찾는 요청은 search_business_report_evidence다. 두 요구가 모두 있으면 둘 다 요청한다.
6. 주가, 매수·매도 추천, 금융업 분석처럼 V1이 지원하지 않는 요청에는 Tool을 호출하지 않는다.
7. 하나의 질문에 필요한 Tool이 여러 개면 모두 요청할 수 있다. 실제 실행 순서는 workflow가 관리한다.

도구 호출 이외의 설명·수치·근거를 만들지 않는다.
'''

# 모든 정답은 개발 평가 전용이며, PROMPT_A/PROMPT_B에는 포함하지 않는다.
EVAL_CASES = [
    {'id': 'F01', 'question': '삼성전자 2024년 매출액과 영업이익을 알려줘.', 'expected_tools': ['get_financial_data'], 'arguments': {'get_financial_data': {'companies': ['삼성전자'], 'years': [2024], 'recent_years': None, 'metrics': ['revenue', 'operating_income']}}},
    {'id': 'F02', 'question': 'SK하이닉스의 2022년부터 2024년까지 매출액 증가율 추이를 보여줘.', 'expected_tools': ['get_financial_data', 'calculate_financial_metric'], 'arguments': {'get_financial_data': {'companies': ['SK하이닉스'], 'years': [2022, 2023, 2024], 'recent_years': None, 'metrics': ['revenue']}, 'calculate_financial_metric': {'companies': ['SK하이닉스'], 'years': [2022, 2023, 2024], 'recent_years': None, 'calculations': ['revenue_growth']}}},
    {'id': 'F03', 'question': '삼성전자와 SK하이닉스의 최근 3년 영업이익률 변화를 비교해줘.', 'expected_tools': ['get_financial_data', 'calculate_financial_metric'], 'arguments': {'get_financial_data': {'companies': ['삼성전자', 'SK하이닉스'], 'years': [], 'recent_years': 3, 'metrics': ['revenue', 'operating_income']}, 'calculate_financial_metric': {'companies': ['삼성전자', 'SK하이닉스'], 'years': [], 'recent_years': 3, 'calculations': ['operating_margin']}}},
    {'id': 'F04', 'question': '현대제철의 2023년 부채비율을 계산해줘.', 'expected_tools': ['get_financial_data', 'calculate_financial_metric'], 'arguments': {'get_financial_data': {'companies': ['현대제철'], 'years': [2023], 'recent_years': None, 'metrics': ['liabilities', 'equity']}, 'calculate_financial_metric': {'companies': ['현대제철'], 'years': [2023], 'recent_years': None, 'calculations': ['debt_ratio']}}},
    {'id': 'F05', 'question': '삼성전자의 2024년 영업활동현금흐름을 확인해줘.', 'expected_tools': ['get_financial_data'], 'arguments': {'get_financial_data': {'companies': ['삼성전자'], 'years': [2024], 'recent_years': None, 'metrics': ['operating_cash_flow']}}},
    {'id': 'F06', 'question': 'LG전자와 삼성전자의 2022~2024년 당기순이익을 비교해줘.', 'expected_tools': ['get_financial_data'], 'arguments': {'get_financial_data': {'companies': ['LG전자', '삼성전자'], 'years': [2022, 2023, 2024], 'recent_years': None, 'metrics': ['net_income']}}},
    {'id': 'D01', 'question': '삼성전자의 2024년 주요 공시 목록을 보여줘.', 'expected_tools': ['search_disclosure_list'], 'arguments': {'search_disclosure_list': {'companies': ['삼성전자'], 'years': [2024], 'recent_years': None}}},
    {'id': 'D02', 'question': 'SK하이닉스가 2023년에 낸 설비투자 관련 공시를 찾아줘.', 'expected_tools': ['search_disclosure_list'], 'arguments': {'search_disclosure_list': {'companies': ['SK하이닉스'], 'years': [2023], 'recent_years': None}}},
    {'id': 'D03', 'question': '현대제철의 최근 2년 공시 목록을 확인해줘.', 'expected_tools': ['search_disclosure_list'], 'arguments': {'search_disclosure_list': {'companies': ['현대제철'], 'years': [], 'recent_years': 2}}},
    {'id': 'R01', 'question': '삼성전자 2024년 사업보고서에서 반도체 사업의 위험요인을 찾아줘.', 'expected_tools': ['search_business_report_evidence'], 'arguments': {'search_business_report_evidence': {'companies': ['삼성전자'], 'years': [2024], 'recent_years': None}}},
    {'id': 'R02', 'question': 'SK하이닉스 2024년 사업보고서에서 HBM 투자 계획의 근거를 찾아줘.', 'expected_tools': ['search_business_report_evidence'], 'arguments': {'search_business_report_evidence': {'companies': ['SK하이닉스'], 'years': [2024], 'recent_years': None}}},
    {'id': 'R03', 'question': '현대제철 2023년 사업보고서에서 공급망 위험 관련 내용을 찾아줘.', 'expected_tools': ['search_business_report_evidence'], 'arguments': {'search_business_report_evidence': {'companies': ['현대제철'], 'years': [2023], 'recent_years': None}}},
    {'id': 'M01', 'question': '삼성전자의 최근 3년 매출과 영업이익률 추세, 그리고 2024 사업보고서의 반도체 위험요인을 함께 정리해줘.', 'expected_tools': ['get_financial_data', 'calculate_financial_metric', 'search_business_report_evidence'], 'arguments': {'get_financial_data': {'companies': ['삼성전자'], 'years': [], 'recent_years': 3, 'metrics': ['revenue', 'operating_income']}, 'calculate_financial_metric': {'companies': ['삼성전자'], 'years': [], 'recent_years': 3, 'calculations': ['operating_margin']}, 'search_business_report_evidence': {'companies': ['삼성전자'], 'years': [2024], 'recent_years': None}}},
    {'id': 'M02', 'question': 'SK하이닉스의 2022~2024년 매출 증가율과 2024년 공시 목록을 같이 확인해줘.', 'expected_tools': ['get_financial_data', 'calculate_financial_metric', 'search_disclosure_list'], 'arguments': {'get_financial_data': {'companies': ['SK하이닉스'], 'years': [2022, 2023, 2024], 'recent_years': None, 'metrics': ['revenue']}, 'calculate_financial_metric': {'companies': ['SK하이닉스'], 'years': [2022, 2023, 2024], 'recent_years': None, 'calculations': ['revenue_growth']}, 'search_disclosure_list': {'companies': ['SK하이닉스'], 'years': [2024], 'recent_years': None}}},
    {'id': 'M03', 'question': '현대제철의 2024년 영업활동현금흐름과 사업보고서의 투자 계획을 함께 조사해줘.', 'expected_tools': ['get_financial_data', 'search_business_report_evidence'], 'arguments': {'get_financial_data': {'companies': ['현대제철'], 'years': [2024], 'recent_years': None, 'metrics': ['operating_cash_flow']}, 'search_business_report_evidence': {'companies': ['현대제철'], 'years': [2024], 'recent_years': None}}},
    {'id': 'X01', 'question': '삼성전자 주식을 지금 사야 할까?', 'expected_tools': [], 'arguments': {}},
    {'id': 'X02', 'question': 'KB금융의 2024년 실적을 분석해줘.', 'expected_tools': [], 'arguments': {}},
]

pd.DataFrame([{'id': case['id'], '질문': case['question'], '정답 Tool': ', '.join(case['expected_tools']) or '없음'} for case in EVAL_CASES])


In [ ]:
llm = ChatOpenAI(
    model=OPENAI_MODEL,
    temperature=TEMPERATURE,
    reasoning_effort='none',
    timeout=REQUEST_TIMEOUT_S,
    max_retries=MAX_RETRIES,
)
tool_router = llm.bind_tools(TOOLS)
PROMPTS = {'A. 간단 Prompt': PROMPT_A, 'B. 상세 정책 Prompt': PROMPT_B}

def normalise_value(value):
    if isinstance(value, list):
        return sorted(str(item).strip() for item in value)
    return value


def normalise_args(tool_name, args):
    args = dict(args or {})
    args.setdefault('years', [])
    args.setdefault('recent_years', None)
    if tool_name == 'get_financial_data':
        args.setdefault('statement_basis', 'DEFAULT')
    return {key: normalise_value(value) for key, value in args.items()}


def usage_from_response(response):
    usage = getattr(response, 'usage_metadata', None) or {}
    metadata = getattr(response, 'response_metadata', None) or {}
    token_usage = metadata.get('token_usage') or {}
    return {
        'input_tokens': usage.get('input_tokens', token_usage.get('prompt_tokens')),
        'output_tokens': usage.get('output_tokens', token_usage.get('completion_tokens')),
        'total_tokens': usage.get('total_tokens', token_usage.get('total_tokens')),
        'model_name': metadata.get('model_name') or metadata.get('model'),
    }


def is_invalid_plan(expected_tools, tool_calls):
    names = [call.get('name') for call in tool_calls]
    if not expected_tools and names:
        return True, '지원 밖 질문에 Tool 호출'
    if expected_tools and not names:
        return True, '지원 질문에 Tool 미호출'
    if len(names) != len(set(names)):
        return True, '동일 Tool 중복 호출'
    if not set(expected_tools).issubset(set(names)):
        return True, '필수 Tool 누락'
    if 'calculate_financial_metric' in names and 'get_financial_data' not in names:
        return True, '재무 원천 조회 없는 계산 Tool 호출'
    for call in tool_calls:
        args = normalise_args(call.get('name'), call.get('args'))
        if call.get('name') not in TOOL_NAMES:
            return True, '등록되지 않은 Tool'
        if not args.get('companies'):
            return True, f'{call.get("name")}: companies 누락'
        if call.get('name') == 'get_financial_data' and not args.get('metrics'):
            return True, 'get_financial_data: metrics 누락'
        if call.get('name') == 'calculate_financial_metric' and not args.get('calculations'):
            return True, 'calculate_financial_metric: calculations 누락'
        if call.get('name') == 'search_business_report_evidence' and not args.get('topics'):
            return True, 'search_business_report_evidence: topics 누락'
    return False, None


def evaluate_case(case, tool_calls, error):
    expected_tools = set(case['expected_tools'])
    actual_names = [call.get('name') for call in tool_calls]
    actual_tools = set(actual_names)
    true_positive = len(expected_tools & actual_tools)
    invalid_plan, invalid_reason = is_invalid_plan(expected_tools, tool_calls) if error is None else (True, '호출 오류')

    argument_results = []
    for tool_name, expected_args in case['arguments'].items():
        actual_call = next((call for call in tool_calls if call.get('name') == tool_name), None)
        if actual_call is None:
            argument_results.append(False)
            continue
        actual_args = normalise_args(tool_name, actual_call.get('args'))
        expected_args = normalise_args(tool_name, expected_args)
        argument_results.append(all(actual_args.get(key) == value for key, value in expected_args.items()))

    return {
        'expected_tool_count': len(expected_tools),
        'actual_tool_call_count': len(actual_names),
        'actual_tool_set_count': len(actual_tools),
        'duplicate_tool_call_count': len(actual_names) - len(actual_tools),
        'true_positive_tools': true_positive,
        'tool_set_exact': actual_tools == expected_tools,
        'argument_exact': all(argument_results) if argument_results else actual_tools == expected_tools,
        'invalid_plan': invalid_plan,
        'invalid_reason': invalid_reason,
        'actual_tool_names': actual_names,
    }


In [ ]:
# Tool은 실행하지 않는다. 각 질문마다 LLM이 반환한 tool_calls만 기록한다.
run_rows = []
total_calls = len(PROMPTS) * len(EVAL_CASES)
call_number = 0

for variant, prompt in PROMPTS.items():
    for case in EVAL_CASES:
        call_number += 1
        print(f'[{call_number}/{total_calls}] {variant} | {case["id"]} 호출 시작')
        started_at = time.perf_counter()
        response, error = None, None
        try:
            response = tool_router.invoke([SystemMessage(prompt), HumanMessage(case['question'])])
            tool_calls = list(getattr(response, 'tool_calls', None) or [])
            usage = usage_from_response(response)
        except Exception as exc:
            tool_calls = []
            usage = {}
            error = f'{type(exc).__name__}: {exc}'
        latency_ms = (time.perf_counter() - started_at) * 1_000
        scores = evaluate_case(case, tool_calls, error)
        run_rows.append({
            'variant': variant, 'case_id': case['id'], 'question': case['question'],
            'expected_tools': case['expected_tools'], 'tool_calls': tool_calls,
            'error': error, 'latency_ms': latency_ms, **usage, **scores,
        })
        print(f'  완료 | tools={scores["actual_tool_names"]} | 오류={error is not None} | {latency_ms:.0f}ms')

results = pd.DataFrame(run_rows)
display(results[['variant', 'case_id', 'expected_tools', 'actual_tool_names', 'tool_set_exact', 'argument_exact', 'invalid_plan', 'latency_ms', 'error']])


In [ ]:
def safe_divide(numerator, denominator):
    return numerator / denominator if denominator else None


summary_rows = []
for variant, group in results.groupby('variant', sort=False):
    tp = int(group['true_positive_tools'].sum())
    actual = int(group['actual_tool_set_count'].sum())
    expected = int(group['expected_tool_count'].sum())
    precision = safe_divide(tp, actual)
    recall = safe_divide(tp, expected)
    tool_f1 = safe_divide(2 * precision * recall, precision + recall) if precision is not None and recall is not None and precision + recall else None
    successful = group[group['error'].isna()]
    summary_rows.append({
        '후보': variant,
        'Tool Precision': precision,
        'Tool Recall': recall,
        'Tool F1': tool_f1,
        'Tool-set Exact Match': group['tool_set_exact'].mean(),
        'Argument Exact Match (계약 필드)': group['argument_exact'].mean(),
        'Invalid-plan Rate': group['invalid_plan'].mean(),
        '유효 Tool 계획률': (~group['invalid_plan']).mean(),
        'API 호출 성공률': group['error'].isna().mean(),
        'p50 latency (ms)': successful['latency_ms'].median() if not successful.empty else None,
        'p95 latency (ms)': successful['latency_ms'].quantile(0.95) if len(successful) >= 2 else None,
        '입력 token 합계': successful['input_tokens'].dropna().sum(),
        '출력 token 합계': successful['output_tokens'].dropna().sum(),
        '호출 오류 수': int(group['error'].notna().sum()),
    })

summary = pd.DataFrame(summary_rows)
display(summary)

# Argument Exact Match (계약 필드)는 각 Tool의 결정론적 평가 필드(기업, 명시/상대 기간, metric·calculation·statement_basis)가 모두 일치한 비율이다.
# 사업보고서 topics·공시 keywords의 의미적 관련성은 이후 RAG/공시 검색 평가에서 별도로 다룬다.
failure_columns = ['variant', 'case_id', 'question', 'expected_tools', 'actual_tool_names', 'tool_calls', 'tool_set_exact', 'argument_exact', 'invalid_plan', 'invalid_reason', 'error']
failures = results[(~results['tool_set_exact']) | (~results['argument_exact']) | results['invalid_plan'] | results['error'].notna()]
if failures.empty:
    print('두 후보 모두 이 개발 평가셋에서 Tool·인자 오류가 없습니다. 품질이 동등하면 비용·지연시간과 실패 사례를 함께 보고 판단합니다.')
else:
    print('아래는 후보 선택 전에 사람이 원문 질문·Tool 요청을 확인해야 할 사례입니다.')
    display(failures[failure_columns])

print('이 실행 결과를 확인한 뒤에만 HTML 결과 보고서와 채택·미채택 판단을 작성합니다.')


## 실행 후 확인할 해석 기준

1. B가 Tool F1·Tool-set Exact Match·Argument Exact Match(계약 필드)에서 명확히 높고 Invalid-plan Rate를 악화시키지 않으면 B를 다음 단계 후보로 둔다.
2. 한 후보의 Precision은 높고 다른 후보의 Recall만 높다면 자동 채택하지 않는다. 누락 Tool과 불필요 Tool의 실제 사용자 영향, 실패 사례를 함께 판단한다.
3. 이 결과가 좋아도 실제 Tool 실행 정확도나 최종 답변 품질을 뜻하지 않는다. 다음 단계에서는 선택한 후보의 Tool 요청을 OpenDART 호출·계산·공시/RAG 경로에 연결해 별도로 평가한다.
